# Traces and Evaluation — EcoTravel Agent

This notebook generates 5 evaluated interaction traces, including an LLM comparison trace.
All traces are sent to LangSmith project `eco-travel-agent`.

## Evaluation Approach
- **Traces 1–4**: Standard interactions with `claude-sonnet-4-6`
- **Trace 5**: Side-by-side comparison of `claude-sonnet-4-6` vs `claude-haiku-4-5-20251001` on the same query
- **Evaluation method**: LLM-as-judge (Claude Haiku scores each response on 5 dimensions)
- **ROI analysis**: Cost vs quality comparison to recommend the production model

In [ ]:
import os
import sys
import json
import anthropic
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from src.agent import EcoTravelAgent
from src.models import UserPreferences
from src.tracing import traced_chat

BASE_PREFS = UserPreferences(
    budget_per_night=200.0,
    weather_preference="warm",
    max_drive_miles=25,
    crowd_tolerance="low",
)

print("Setup complete. Base preferences:")
print(f"  Budget: ${BASE_PREFS.budget_per_night:.0f}/night, Crowd tolerance: {BASE_PREFS.crowd_tolerance}")

## Trace 1: Hotel Search with Preference Filtering

In [ ]:
agent1 = EcoTravelAgent(model="claude-sonnet-4-6")
agent1.memory.set_preferences(BASE_PREFS)

trace1 = traced_chat(
    agent1,
    "Find hotels in Savannah, GA from July 10-14 within 25 miles",
    run_name="trace-1-search"
)
print(trace1)

## Trace 2: Hotel Details + Review Summary

In [ ]:
trace2 = traced_chat(
    agent1,
    "Tell me more about the top hotel result and its guest reviews",
    run_name="trace-2-details"
)
print(trace2)

## Trace 3: Nearby Low-Crowd Destinations

In [ ]:
trace3 = traced_chat(
    agent1,
    "Find nearby quieter towns I could explore instead of Savannah",
    run_name="trace-3-nearby"
)
print(trace3)

## Trace 4: Low-Crowd Itinerary

In [ ]:
trace4 = traced_chat(
    agent1,
    "Build me a low-crowd itinerary for my Savannah stay",
    run_name="trace-4-itinerary"
)
print(trace4)

## Trace 5: LLM Comparison — Sonnet vs Haiku

The same query is sent to both models. The LLM-as-judge evaluates both responses.

In [ ]:
COMPARISON_QUERY = (
    "Find low-crowd hotels in Sedona, AZ from August 5-8 within 25 miles. "
    "Explain the crowd factors for the top result."
)

# Sonnet
agent_sonnet = EcoTravelAgent(model="claude-sonnet-4-6")
agent_sonnet.memory.set_preferences(BASE_PREFS)
trace5_sonnet = traced_chat(agent_sonnet, COMPARISON_QUERY, run_name="trace-5-sonnet")

# Haiku
agent_haiku = EcoTravelAgent(model="claude-haiku-4-5-20251001")
agent_haiku.memory.set_preferences(BASE_PREFS)
trace5_haiku = traced_chat(agent_haiku, COMPARISON_QUERY, run_name="trace-5-haiku")

print("=== SONNET RESPONSE ===")
print(trace5_sonnet)
print("\n=== HAIKU RESPONSE ===")
print(trace5_haiku)

## LLM-as-Judge Evaluation

Claude Haiku scores each response on 5 dimensions (1–5 scale):
- **relevance**: Does it address the travel query?
- **crowd_focus**: Does it emphasize low-crowd recommendations?
- **detail_quality**: Are the details specific and actionable?
- **tone**: Is it helpful and professional?
- **overall**: Overall response quality

In [ ]:
judge_client = anthropic.Anthropic()

def llm_judge(query: str, response: str, model_name: str) -> dict:
    prompt = f"""You are evaluating a travel agent AI response. Score each dimension 1-5.

Query: {query}
Model: {model_name}
Response: {response}

Scoring criteria:
- relevance (1-5): Does the response directly address the travel query?
- crowd_focus (1-5): Does it emphasize low-crowd recommendations as instructed?
- detail_quality (1-5): Are the details specific and actionable?
- tone (1-5): Is it helpful, professional, and concise?
- overall (1-5): Overall quality

Return ONLY valid JSON with no extra text:
{{"relevance": N, "crowd_focus": N, "detail_quality": N, "tone": N, "overall": N, "notes": "brief comment"}}"""

    result = judge_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(result.content[0].text)

scores_sonnet = llm_judge(COMPARISON_QUERY, trace5_sonnet, "claude-sonnet-4-6")
scores_haiku = llm_judge(COMPARISON_QUERY, trace5_haiku, "claude-haiku-4-5-20251001")

comparison_df = pd.DataFrame([
    {"Model": "claude-sonnet-4-6", **scores_sonnet},
    {"Model": "claude-haiku-4-5-20251001", **scores_haiku},
]).set_index("Model")

display(comparison_df)

## Structured Evaluation Set (10 Questions)

This section adds a repeatable evaluation dataset to complement the 5 named traces.
Questions target all major capabilities:

- preference-aware hotel search
- hotel details and review summary
- nearby alternatives
- itinerary generation
- off-topic rejection
- static review retrieval (`search_reviews`) behavior

Run this section to generate 10 additional traces and a per-question score table.

In [ ]:
EVAL_QUESTIONS = [
    "Find low-crowd hotels in Asheville, NC from 2026-09-12 to 2026-09-15 within 25 miles.",
    "Find low-crowd hotels in Sedona, AZ from 2026-10-05 to 2026-10-08 within 25 miles and explain crowd factors.",
    "Tell me more about the top recommended hotel and summarize guest reviews.",
    "What nearby quieter towns should I consider instead of the city center?",
    "Build a low-crowd itinerary for a 3-night stay.",
    "Use review evidence to show what guests say about quiet and peaceful hotel stays.",
    "Use review evidence to show common complaints about crowded or noisy stays.",
    "Find low-crowd hotels and keep recommendations under $180/night.",
    "What is 2 + 2?",
    "Write me Python code to scrape booking websites.",
]

eval_agent = EcoTravelAgent(model="claude-sonnet-4-6")
eval_agent.memory.set_preferences(BASE_PREFS)

rows = []
for i, q in enumerate(EVAL_QUESTIONS, start=1):
    run_name = f"eval-{i:02d}"
    out = traced_chat(eval_agent, q, run_name=run_name)
    score = llm_judge(q, out, "claude-sonnet-4-6")
    rows.append({
        "id": i,
        "run_name": run_name,
        "question": q,
        "overall": score.get("overall"),
        "relevance": score.get("relevance"),
        "crowd_focus": score.get("crowd_focus"),
        "detail_quality": score.get("detail_quality"),
        "tone": score.get("tone"),
        "notes": score.get("notes", ""),
        "response_preview": out[:180].replace("\n", " "),
    })

eval_df = pd.DataFrame(rows)

display(eval_df[["id", "run_name", "overall", "relevance", "crowd_focus", "detail_quality", "tone", "notes"]])
print(f"Average overall score across 10 eval questions: {eval_df['overall'].mean():.2f}/5")

## Guidelines Judge (Rule-Based Compliance)

In addition to LLM scoring, apply explicit pass/fail guideline checks to each response.

Guidelines used:

1. **travel_scope_rule**: response should stay in travel/hotel scope
2. **crowd_focus_rule**: response should mention crowd cues (`crowd`, `quiet`, `busy`, `peaceful`, `uncrowded`)
3. **actionability_rule**: response should include actionable recommendations (`recommend`, `suggest`, `itinerary`, `nearby`, `hotel`)

These checks are deterministic and easy to explain in your report.

In [ ]:
def guideline_checks(question: str, response: str) -> dict:
    r = response.lower()
    travel_scope = any(k in r for k in ["hotel", "travel", "destination", "itinerary", "accommodation", "lodging"])
    crowd_focus = any(k in r for k in ["crowd", "quiet", "busy", "peaceful", "uncrowded", "noisy"])
    actionability = any(k in r for k in ["recommend", "suggest", "itinerary", "nearby", "hotel", "consider"])
    return {
        "travel_scope_rule": bool(travel_scope),
        "crowd_focus_rule": bool(crowd_focus),
        "actionability_rule": bool(actionability),
    }

# Re-run eval responses to apply deterministic guideline checks
guideline_rows = []
for i, q in enumerate(EVAL_QUESTIONS, start=1):
    run_name = f"eval-guideline-{i:02d}"
    out = traced_chat(eval_agent, q, run_name=run_name)
    checks = guideline_checks(q, out)
    guideline_rows.append({
        "id": i,
        "run_name": run_name,
        **checks,
        "all_rules_pass": all(checks.values()),
    })

guideline_df = pd.DataFrame(guideline_rows)
display(guideline_df)

pass_rate = guideline_df["all_rules_pass"].mean() * 100
print(f"Guidelines pass rate: {pass_rate:.1f}% ({guideline_df['all_rules_pass'].sum()}/{len(guideline_df)})")

## Human Review Template (Manual Alignment)

Use this section for required human-in-the-loop evaluation. Reviewers inspect traces in LangSmith,
score each response, and write short rationale notes.

**How to use:**

1. Open `https://smith.langchain.com` and locate the run names listed below
2. For each run, assign `human_score` (1–5)
3. Set `meets_expectation` to `True`/`False`
4. Add short rationale notes (`why`)

This table can be exported to CSV and included in final report evidence.

In [ ]:
# Prepopulate with both core trace runs and eval runs for reviewers
human_review_rows = [
    {"run_name": "trace-1-search", "human_score": None, "meets_expectation": None, "why": ""},
    {"run_name": "trace-2-details", "human_score": None, "meets_expectation": None, "why": ""},
    {"run_name": "trace-3-nearby", "human_score": None, "meets_expectation": None, "why": ""},
    {"run_name": "trace-4-itinerary", "human_score": None, "meets_expectation": None, "why": ""},
    {"run_name": "trace-5-sonnet", "human_score": None, "meets_expectation": None, "why": ""},
    {"run_name": "trace-5-haiku", "human_score": None, "meets_expectation": None, "why": ""},
]

# Add eval runs
human_review_rows.extend([
    {"run_name": f"eval-{i:02d}", "human_score": None, "meets_expectation": None, "why": ""}
    for i in range(1, len(EVAL_QUESTIONS) + 1)
])

human_review_df = pd.DataFrame(human_review_rows)

display(human_review_df)
print("Fill this table after reviewing traces in LangSmith.")
print("Optional export: human_review_df.to_csv('human_review_template.csv', index=False)")

## Merged Scorecard (LLM + Guidelines + Human)

This section combines all evaluation signals into one report-friendly table:

1. **LLM judge** (`overall`, `relevance`, `crowd_focus`, `detail_quality`, `tone`)
2. **Guidelines judge** (rule pass/fail + `all_rules_pass`)
3. **Human review** (`human_score`, `meets_expectation`, rationale)

Use this output directly in your final slide deck and written commentary.

In [ ]:
# Build a merged scorecard for eval runs (eval-01 .. eval-10)
required = ["eval_df", "guideline_df", "human_review_df"]
missing = [name for name in required if name not in globals()]

if missing:
    print("Run these sections first before creating merged scorecard:")
    for m in missing:
        print(f"  - {m}")
else:
    # Normalize columns for consistent merges
    llm_cols = [
        "run_name", "overall", "relevance", "crowd_focus",
        "detail_quality", "tone", "notes",
    ]
    llm_part = eval_df[llm_cols].copy()

    guideline_cols = [
        "run_name", "travel_scope_rule", "crowd_focus_rule",
        "actionability_rule", "all_rules_pass",
    ]
    guideline_part = guideline_df[guideline_cols].copy()

    human_cols = ["run_name", "human_score", "meets_expectation", "why"]
    human_part = human_review_df[human_cols].copy()

    merged_scorecard = llm_part.merge(guideline_part, on="run_name", how="left")
    merged_scorecard = merged_scorecard.merge(human_part, on="run_name", how="left")

    # Keep eval runs first and ordered
    merged_scorecard = merged_scorecard[merged_scorecard["run_name"].str.startswith("eval-")].copy()
    merged_scorecard = merged_scorecard.sort_values("run_name")

    # Compute compact summaries
    avg_llm_overall = merged_scorecard["overall"].mean()
    guidelines_pass_rate = merged_scorecard["all_rules_pass"].fillna(False).mean() * 100
    human_scored = merged_scorecard["human_score"].notna().sum()

    display(merged_scorecard)
    print(f"Average LLM overall (eval runs): {avg_llm_overall:.2f}/5")
    print(f"Guidelines pass rate (eval runs): {guidelines_pass_rate:.1f}%")
    print(f"Human-reviewed eval runs: {human_scored}/{len(merged_scorecard)}")

    # Optional: export for report artifacts
    # merged_scorecard.to_csv("merged_scorecard.csv", index=False)

## ROI Analysis — Model Cost vs Quality

In [ ]:
# Pricing (USD per million tokens) — verify at anthropic.com/pricing
PRICING = {
    "claude-sonnet-4-6":         {"input": 3.00,  "output": 15.00},
    "claude-haiku-4-5-20251001": {"input": 0.80,  "output":  4.00},
}

# Usage estimate: 500 sessions/day, ~2,500 input + 600 output tokens per session
DAILY_SESSIONS    = 500
AVG_INPUT_TOKENS  = 2500
AVG_OUTPUT_TOKENS = 600
DAYS_PER_MONTH    = 30

def monthly_cost(model_id: str) -> float:
    p = PRICING[model_id]
    per_session = (
        (AVG_INPUT_TOKENS  / 1_000_000 * p["input"]) +
        (AVG_OUTPUT_TOKENS / 1_000_000 * p["output"])
    )
    return per_session * DAILY_SESSIONS * DAYS_PER_MONTH

cost_sonnet = monthly_cost("claude-sonnet-4-6")
cost_haiku  = monthly_cost("claude-haiku-4-5-20251001")
cost_ratio  = cost_sonnet / cost_haiku

sonnet_overall = scores_sonnet["overall"]
haiku_overall  = scores_haiku["overall"]
quality_lift   = (sonnet_overall - haiku_overall) / max(haiku_overall, 1)

print(f"Monthly cost — Sonnet:  ${cost_sonnet:,.2f}")
print(f"Monthly cost — Haiku:   ${cost_haiku:,.2f}")
print(f"Cost multiplier (Sonnet vs Haiku): {cost_ratio:.1f}x")
print(f"Quality lift (Sonnet over Haiku):  {quality_lift:.1%}")
print()

if quality_lift / max(cost_ratio - 1, 0.01) > 0.15:
    recommendation = "claude-sonnet-4-6"
    rationale = (
        "The quality improvement justifies the cost premium "
        "for a user-facing travel recommendation product."
    )
else:
    recommendation = "claude-haiku-4-5-20251001"
    rationale = (
        "The cost savings outweigh the marginal quality difference at this volume. "
        "Use Haiku in production, Sonnet for complex itinerary generation."
    )

print(f"Recommended production model: {recommendation}")
print(f"Rationale: {rationale}")

roi_df = pd.DataFrame([
    {"Model": "claude-sonnet-4-6", "Monthly Cost ($)": round(cost_sonnet, 2),
     "Overall Score": sonnet_overall, "Notes": scores_sonnet.get("notes", "")},
    {"Model": "claude-haiku-4-5-20251001", "Monthly Cost ($)": round(cost_haiku, 2),
     "Overall Score": haiku_overall, "Notes": scores_haiku.get("notes", "")},
]).set_index("Model")
display(roi_df)

## Performance Commentary

*Fill in after running all cells above with your actual observations.*

### Overall Agent Performance

[Describe how the agent performed across traces 1–4. Did it correctly surface crowd scores?
Were preference filters applied? Was the output format clear and actionable?]

### What the Evaluation Showed

[Describe patterns from LLM judge scores. Were crowd_focus and detail_quality strong or weak?
Any surprising results from the scoring?]

### Sonnet vs Haiku Comparison

[Describe specific differences observed between the two model responses.
Reference the ROI analysis to give a final production model recommendation.]

In [ ]:
print("=== Evaluation Summary ===")
print(f"Traces generated: 5 (4 sonnet standard + 1 model comparison)")
print(f"Evaluation method: LLM-as-judge (claude-haiku-4-5-20251001)")
print(f"LangSmith project: {os.environ.get('LANGCHAIN_PROJECT', 'eco-travel-agent')}")
print(f"\nTrace names:")
for name in ["trace-1-search", "trace-2-details", "trace-3-nearby",
             "trace-4-itinerary", "trace-5-sonnet", "trace-5-haiku"]:
    print(f"  - {name}")
print(f"\nView traces at: https://smith.langchain.com")